### Random Forest Algorithm

#### 1.Import the libraries.

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

#### 2. Import the dataset

In [2]:
df = pd.read_csv("diabetess.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [7]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,121.656250,72.386719,27.334635,94.652344,32.450911,0.471876,33.240885,0.348958
std,3.369578,30.438286,12.096642,9.229014,105.547598,6.875366,0.331329,11.760232,0.476951
min,0.000000,44.000000,24.000000,7.000000,14.000000,18.200000,0.078000,21.000000,0.000000
25%,1.000000,99.750000,64.000000,23.000000,30.500000,27.500000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,31.250000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [8]:
df['Glucose'].replace(0,np.median(df['Glucose']),inplace=True)
df['BloodPressure'].replace(0,np.median(df['BloodPressure']),inplace=True)
df['SkinThickness'].replace(0,np.median(df['SkinThickness']),inplace=True)
df['Insulin'].replace(0,np.median(df['Insulin']),inplace=True)
df['BMI'].replace(0,np.median(df['BMI']),inplace=True)


#### 3. Assigning Feature Variable to X and Target variable to y

In [9]:
# Putting feature variable to X
X = df.drop('Outcome', axis=1)

# Putting target variable to y
y = df['Outcome']

In [10]:
y.value_counts()

Outcome
0    500
1    268
Name: count, dtype: int64

#### 4. Perform Train-Test-Split

In [11]:
# lets split the data into train and test
from sklearn.model_selection import train_test_split

# Splitting the data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [12]:
X_train.shape, X_test.shape

((537, 8), (231, 8))

#### 5. Import Random Forest Classifier and fit the data

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1, max_depth=10,
                                       n_estimators=100, oob_score=True)
rf_classifier.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_jobs=-1, oob_score=True, random_state=42)

In [14]:
# checking the oob score
rf_classifier.oob_score_

0.7579143389199255

#### 6. Hyperparameter tuning for Random Forest using GridSearchCV

In [15]:
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

params = {
    'max_depth': [2,3,5,10,20,25],
    'min_samples_leaf': [5,10,20,50,100,200,250],
    'n_estimators': [10,25,30,50,100,150,200]
}

from sklearn.model_selection import GridSearchCV

# Instantiate the grid search model
gscv = GridSearchCV(estimator=rf,
                           param_grid=params,
                           cv = 5,
                           n_jobs=-1, verbose=1, scoring="accuracy")

gscv.fit(X_train, y_train)

Fitting 5 folds for each of 294 candidates, totalling 1470 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [2, 3, 5, 10, 20, 25],
                         'min_samples_leaf': [5, 10, 20, 50, 100, 200, 250],
                         'n_estimators': [10, 25, 30, 50, 100, 150, 200]},
             scoring='accuracy', verbose=1)

In [16]:
gscv.best_score_

0.7820699203876774

In [17]:
gscv.best_params_

{'max_depth': 20, 'min_samples_leaf': 5, 'n_estimators': 50}

In [18]:
rf_best_model_grid = gscv.best_estimator_
rf_best_model_grid

RandomForestClassifier(max_depth=20, min_samples_leaf=5, n_estimators=50,
                       n_jobs=-1, random_state=42)

#### 7. Visualization

In [19]:
from sklearn.tree import plot_tree
plt.figure(figsize=(80,40))
plot_tree(rf_best_model.estimators_[5], feature_names = X.columns,class_names=['Diabetes', "No Diabetes"],filled=True);
plt.savefig("RF_diabetes_est5.png")

NameError: name 'rf_best_model' is not defined

<Figure size 8000x4000 with 0 Axes>

In [ ]:
from sklearn.tree import plot_tree
plt.figure(figsize=(80,40))
plot_tree(rf_best_model.estimators_[7], feature_names = X.columns,class_names=['Diabetes', "No Diabetes"],filled=True);
plt.savefig("RF_diabetes_est7.png")

In [ ]:
pwd

#### 8. Sorting the data with the help of feature importance

In [ ]:
rf_best_model.feature_importances_

In [20]:
imp_df = pd.DataFrame({
    "Varname": X_train.columns,
    "Imp": rf_best_model.feature_importances_
})
imp_df

imp_df.sort_values(by="Imp", ascending=False, ignore_index=True)

NameError: name 'rf_best_model' is not defined

#### Key Takeaways

In [21]:
y_pred = rf_classifier.predict(X_test)

In [22]:
y_test, len(y_test)

(668    0
 324    0
 624    0
 690    0
 473    0
       ..
 619    1
 198    1
 538    0
 329    0
 302    0
 Name: Outcome, Length: 231, dtype: int64,
 231)

In [23]:
from sklearn.metrics import accuracy_score, classification_report

In [24]:
y_pred

array([0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1,
       0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1,
       0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
       0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0,
       0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], dtype=int64)

In [29]:
y_pred2

array([1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1,
       0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0,
       0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1,
       1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], dtype=int64)

In [26]:
accuracy = accuracy_score(y_pred, y_test)
print(accuracy)

0.7532467532467533


In [27]:
#accurracy for gscv
y_pred2 = rf_best_model_grid.predict(X_test)

In [28]:
accuracy_grid = accuracy_score(y_pred2, y_test)
print(accuracy_grid)

0.7532467532467533


In [60]:
# Randomized search cv

In [39]:
rf1 = RandomForestClassifier(random_state=42, n_jobs=-1)

params = {
    'max_depth': np.arange(1,15),
    'min_samples_leaf': [5,10,20,50,100,200,250],
    'n_estimators': [10,25,30,50,100,150,200]
}

from sklearn.model_selection import RandomizedSearchCV

# Instantiate the Randomized search model
rscv = RandomizedSearchCV(estimator=rf1,
                           param_distributions=params,
                           cv = 5,
                           n_jobs=-1, verbose=1, scoring="accuracy")

rscv.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
                   n_jobs=-1,
                   param_distributions={'max_depth': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14]),
                                        'min_samples_leaf': [5, 10, 20, 50, 100,
                                                             200, 250],
                                        'n_estimators': [10, 25, 30, 50, 100,
                                                         150, 200]},
                   scoring='accuracy', verbose=1)

In [40]:
rscv.best_score_

0.7615957078573901

In [41]:
rscv.best_params_

{'n_estimators': 100, 'min_samples_leaf': 50, 'max_depth': 9}

In [42]:
a = np.arange(1,15)
a

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14])

In [43]:
rf_best_model1= rscv.best_estimator_
rf_best_model1

RandomForestClassifier(max_depth=9, min_samples_leaf=50, n_jobs=-1,
                       random_state=42)

In [44]:
y_pred1 = rf_best_model1.predict(X_test)

In [45]:
accuracy_r = accuracy_score(y_pred1, y_test)
print(accuracy_r)

0.7402597402597403


In [51]:
report=classification_report(y_pred1, y_test)
print(report)

              precision    recall  f1-score   support

           0       0.86      0.77      0.81       169
           1       0.51      0.66      0.58        62

    accuracy                           0.74       231
   macro avg       0.69      0.72      0.69       231
weighted avg       0.77      0.74      0.75       231

